# Notebook 02 — Training the BiEncoder

**Run this on Google Colab with a T4 GPU.**  
`Runtime → Change runtime type → T4 GPU`

## What happens in this notebook
1. Load training pairs generated in Notebook 01
2. Fine-tune `distilbert-base-uncased` with InfoNCE contrastive loss
3. Plot training and validation loss curves
4. Save the best checkpoint to Google Drive

## Training setup
| Hyperparameter | Value | Why |
|---|---|---|
| Loss | InfoNCELoss | In-batch negatives — batch of 32 gives 31 free negatives per query |
| Temperature | 0.07 | Standard value from SimCSE / MoCo literature |
| Batch size | 32 | Larger batch → more negatives per step (better signal) |
| Learning rate | 2e-5 | Standard fine-tuning LR for BERT-family models |
| LR schedule | Linear warmup (10%) + linear decay | Prevents large gradient steps early |
| Epochs | 5 | ~5 min on T4. Early stopping on val loss |
| Gradient clipping | 1.0 | Prevents gradient explosion in the first epoch |

---
## Step 1 — Setup (Colab + Drive)

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Neural_Search_Engine'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q transformers tokenizers safetensors rank-bm25
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

import torch
print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))
    print('VRAM     :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

---
## Step 2 — Load Training Data

In [ ]:
import json

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

train_pairs = load_json('data/processed/train_pairs.json')
val_pairs   = load_json('data/processed/val_pairs.json')

print(f'Train pairs : {len(train_pairs)}')
print(f'Val pairs   : {len(val_pairs)}')

# Preview one pair
p = train_pairs[0]
print(f"\nExample query    : {p['query']}")
print(f"Example positive : {p['positive_text'][:120]}...")

---
## Step 3 — Initialise Model and Trainer

In [ ]:
from src.model import BiEncoder
from src.train import Trainer

# ── Hyperparameters ─────────────────────────────────────────────────────────
BATCH_SIZE  = 32    # larger = more in-batch negatives = better InfoNCE signal
EPOCHS      = 5
LR          = 2e-5  # standard BERT fine-tuning learning rate
TEMPERATURE = 0.07  # InfoNCE temperature — lower = harder, sharper distribution
MAX_LENGTH  = 256   # must match what the data was chunked to
CHECKPOINT  = 'checkpoints'

model = BiEncoder()  # loads distilbert-base-uncased weights

trainer = Trainer(
    model       = model,
    train_pairs = train_pairs,
    val_pairs   = val_pairs,
    output_dir  = CHECKPOINT,
    batch_size  = BATCH_SIZE,
    epochs      = EPOCHS,
    lr          = LR,
    temperature = TEMPERATURE,
    max_length  = MAX_LENGTH,
)

print('Trainer ready.')
print(f'Batches per epoch : {len(trainer.train_loader)}')
print(f'Total steps       : {len(trainer.train_loader) * EPOCHS}')

---
## Step 4 — Train

Expected time on T4 GPU: **~5–8 minutes** for 5 epochs.  
Loss should start near `log(batch_size)` ≈ 3.47 and decrease steadily.

In [ ]:
import time

t0 = time.time()
history = trainer.fit()
print(f'\nTotal training time: {(time.time()-t0)/60:.1f} min')

---
## Step 5 — Plot Learning Curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

epochs_x = list(range(1, len(history['train_loss']) + 1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_x, history['train_loss'], marker='o', label='Train loss', color='steelblue')
ax.plot(epochs_x, history['val_loss'],   marker='s', label='Val loss',   color='tomato')
ax.axhline(y=min(history['val_loss']), linestyle='--', color='tomato', alpha=0.4, label='Best val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('BiEncoder Training Curves')
ax.legend()
ax.grid(alpha=0.3)

# Add y-reference: random loss ≈ log(batch_size)
import math
random_loss = math.log(BATCH_SIZE)
ax.axhline(y=random_loss, linestyle=':', color='grey', alpha=0.6, label=f'Random baseline log(B)={random_loss:.2f}')
ax.legend()
plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150)
plt.show()

print(f'Best val loss : {min(history["val_loss"]):.4f}')
print(f'Random loss   : {random_loss:.4f}   (this is what an untrained model achieves)')
print(f'Improvement   : {random_loss - min(history["val_loss"]):.4f}')

In [ ]:
# Print the full log table
log_df = pd.read_csv('checkpoints/training_log.csv')
print(log_df.to_string(index=False))

---
## Step 6 — Verify the Saved Checkpoint

In [ ]:
import os
for fname in os.listdir('checkpoints'):
    fpath = os.path.join('checkpoints', fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<30} {size_mb:.1f} MB')

# Quick load test
model_loaded = BiEncoder()
model_loaded.load_state_dict(torch.load('checkpoints/best.pt', map_location='cpu'))
model_loaded.eval()
print('\nCheckpoint loads correctly ✓')

---
Training complete. Proceed to **03_evaluation.ipynb** to compare with BM25.